# 🌞 Solar Panel Fault Detection - CNN (ResNet)

**Convolutional Neural Network for Comparison**

**Goal:** Compare CNN vs Vision Transformer (86%)

---

## 1. Install Packages

In [1]:
print("Installing packages...")
import sys
!{sys.executable} -m pip install timm albumentations -q
print("✓ Packages installed!")

Installing packages...

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
✓ Packages installed!


## 2. Import Libraries

In [2]:
import os
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/home/compute.ashesi.lan/e.bilson/python-env/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Libraries imported!
PyTorch: 2.9.1+cu128
CUDA: True
GPU: Quadro RTX 6000


## 3. Configuration

In [18]:
# Paths - CORRECTED!
TRAIN_PATH = "/home/compute.ashesi.lan/e.bilson/Fault_detect_V2/Solar_V2/data_v2/images/train/"
TEST_PATH = "/home/compute.ashesi.lan/e.bilson/Fault_detect_V2/Solar_V2/data_v2/images/test/"

# Classes
CLASS_NAMES = [
    'Cell-Fault', 'Cracking', 'Diode-Fault', 'Hot-Spot',
    'No-Anomaly', 'Offline-Module', 'Shadowing', 'Soiling', 'Vegetation'
]

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_CLASSES = len(CLASS_NAMES)

import torch
import numpy as np
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

print("="*80)
print("CONFIGURATION - CNN (ResNet)")
print("="*80)
print(f"Train path: {TRAIN_PATH}")
print(f"Test path: {TEST_PATH}")

# CHECK IF PATHS EXIST
if os.path.exists(TRAIN_PATH):
    print("OK - Train path EXISTS!")
    folders = os.listdir(TRAIN_PATH)
    print(f"  Found {len(folders)} class folders: {folders[:3]}...")
else:
    print("ERROR - Train path DOES NOT EXIST!")

CONFIGURATION - CNN (ResNet)
Train path: /home/compute.ashesi.lan/e.bilson/Fault_detect_V2/Solar_V2/data_v2/images/train/
Test path: /home/compute.ashesi.lan/e.bilson/Fault_detect_V2/Solar_V2/data_v2/images/test/
OK - Train path EXISTS!
  Found 9 class folders: ['Soiling', 'Shadowing', 'Offline-Module']...


## 4. Data Augmentation (Same as ViT)

In [19]:
# Same augmentation as Vision Transformer for fair comparison
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.GaussianBlur(p=0.3),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2()
])

test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2()
])

print("✓ Augmentation defined!")

✓ Augmentation defined!


## 5. Dataset

In [25]:
import os
import cv2
from torch.utils.data import Dataset

class SolarPanelDataset(Dataset):
    def __init__(self, data_path, class_names, transform=None):
        self.data_path = data_path
        self.class_names = class_names
        self.transform = transform
        self.samples = []
        
        for class_idx, class_name in enumerate(class_names):
            class_folder = os.path.join(data_path, class_name)
            if not os.path.exists(class_folder):
                continue
            
            image_files = [f for f in os.listdir(class_folder) 
                          if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif'))]
            
            for img_file in image_files:
                img_path = os.path.join(class_folder, img_file)
                self.samples.append((img_path, class_idx))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, label

print("Dataset class ready")

Dataset class ready


## 6. Load Data

In [26]:
print("="*80)
print("LOADING DATA")
print("="*80)

train_dataset = SolarPanelDataset(TRAIN_PATH, CLASS_NAMES, train_transform)
test_dataset = SolarPanelDataset(TEST_PATH, CLASS_NAMES, test_transform)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train: {train_size} | Val: {val_size} | Test: {len(test_dataset)}")
print("✓ Data loaded!")

LOADING DATA


ValueError: num_samples should be a positive integer value, but got num_samples=0

## 7. CNN Model (ResNet-18)

In [ ]:
# Create ResNet-18 model (pre-trained on ImageNet)
model = timm.create_model('resnet18', pretrained=True, num_classes=NUM_CLASSES)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*80)
print("CNN MODEL - ResNet-18")
print("="*80)
print(f"Architecture: ResNet-18 (CNN)")
print(f"Pre-trained: ImageNet")
print(f"Total parameters: {total_params:,}")
print(f"Trainable: {trainable_params:,}")
print(f"\n✓ Model created!")

## 8. Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

print("✓ Training setup complete!")

## 9. Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, scaler=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        if scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': running_loss/len(pbar), 'acc': 100.*correct/total})
    
    return running_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

print("✓ Training functions defined!")

## 10. Train CNN

In [ ]:
print("="*80)
print("TRAINING CNN (ResNet-18)")
print("="*80)
print("This will take 3-5 minutes...\n")

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience = 10
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_cnn_model.pth')
        print(f"✓ New best! Val Acc: {val_acc:.2f}%")
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\n{'='*80}")
print(f"✓ Training Complete! Best Val Acc: {best_val_acc:.2f}%")
print(f"{'='*80}")

model.load_state_dict(torch.load('best_cnn_model.pth'))

## 11. Evaluate CNN

In [ ]:
print("\n" + "="*80)
print("EVALUATING CNN ON TEST SET")
print("="*80)

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

test_acc = accuracy_score(all_labels, all_preds)

print(f"\n{'='*80}")
print(f"CNN TEST ACCURACY: {test_acc*100:.2f}%")
print(f"{'='*80}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

cm = confusion_matrix(all_labels, all_preds)

## 12. Visualize Training

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(history['train_loss'], label='Train', linewidth=2)
ax1.plot(history['val_loss'], label='Val', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('CNN Training Loss', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], label='Train', linewidth=2)
ax2.plot(history['val_acc'], label='Val', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('CNN Training Accuracy', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_training.png', dpi=150)
print("✓ Saved: cnn_training.png")
plt.show()

## 13. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, cbar_kws={'label': 'Count'})

ax.set_title('Confusion Matrix - CNN (ResNet-18)', fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel('True', fontsize=12)
ax.set_xlabel('Predicted', fontsize=12)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=150)
print("✓ Saved: cnn_confusion_matrix.png")
plt.show()

## 14. CNN vs Vision Transformer Comparison

In [ ]:
print("\n" + "="*80)
print("CNN vs VISION TRANSFORMER COMPARISON")
print("="*80)

# Update with your Vision Transformer result
VIT_ACCURACY = 86.0  # Your Vision Transformer result

results = {
    'YOLOv8 (v1.0)': 77.4,
    'YOLOv8 (v2.0)': 77.0,
    'CNN (ResNet-18)': test_acc * 100,
    'Vision Transformer': VIT_ACCURACY,
    'Paper (ResNet Ensemble)': 85.9
}

print("\nResults:")
for method, acc in results.items():
    print(f"  {method:25s}: {acc:6.2f}%")

# Determine winner
winner = 'CNN' if test_acc*100 > VIT_ACCURACY else 'Vision Transformer'
difference = abs(test_acc*100 - VIT_ACCURACY)

print(f"\n🏆 WINNER: {winner} (by {difference:.1f}%)")

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 7))
methods = list(results.keys())
accuracies = list(results.values())
colors = ['skyblue', 'lightcoral', 'orange', 'lightgreen', 'gold']

bars = ax.bar(methods, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=2)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('CNN vs Transformer: All Methods Compared', fontsize=15, fontweight='bold', pad=20)
ax.set_ylim([0, 100])
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=85, color='green', linestyle='--', alpha=0.5, label='Target: 85%')
ax.legend()
plt.xticks(rotation=15, ha='right')

plt.tight_layout()
plt.savefig('cnn_vs_transformer.png', dpi=150)
print("\n✓ Saved: cnn_vs_transformer.png")
plt.show()

## 15. Final Summary

In [ ]:
print("\n" + "="*80)
print("✓✓✓ CNN TRAINING COMPLETE! ✓✓✓")
print("="*80)

print("\n📊 CNN RESULTS:")
print(f"  Test Accuracy: {test_acc*100:.2f}%")
print(f"  Best Val Acc: {best_val_acc:.2f}%")
print(f"  Epochs: {len(history['train_loss'])}")
print(f"  Parameters: {total_params:,}")

print("\n📊 VISION TRANSFORMER RESULTS:")
print(f"  Test Accuracy: {VIT_ACCURACY:.2f}%")

print("\n🏆 COMPARISON:")
if test_acc*100 > VIT_ACCURACY:
    print(f"  CNN WINS by {test_acc*100 - VIT_ACCURACY:.1f}%!")
    print("  → CNNs are better for this task!")
elif test_acc*100 < VIT_ACCURACY:
    print(f"  Vision Transformer WINS by {VIT_ACCURACY - test_acc*100:.1f}%!")
    print("  → Transformers are better for this task!")
else:
    print("  TIE! Both architectures perform equally well!")

print(f"\n  CNN vs YOLOv8: +{test_acc*100 - 77.0:.1f}%")
print(f"  CNN vs Paper: {'+' if test_acc*100 > 85.9 else ''}{test_acc*100 - 85.9:.1f}%")

print("\n📁 FILES CREATED:")
print("  ✓ cnn_training.png")
print("  ✓ cnn_confusion_matrix.png")
print("  ✓ cnn_vs_transformer.png")
print("  ✓ best_cnn_model.pth")

print("\n💡 CONCLUSION:")
best_model = 'CNN (ResNet-18)' if test_acc*100 > VIT_ACCURACY else 'Vision Transformer'
best_acc = max(test_acc*100, VIT_ACCURACY)
print(f"  Best model: {best_model}")
print(f"  Best accuracy: {best_acc:.2f}%")
print(f"  Recommendation: Use {best_model} for deployment!")

print("\n" + "="*80)
print("Now you can compare CNN vs Transformer!")
print("="*80)